### Preparing dataset and csv files for training and validation purposes

In [2]:
import csv
import os
import torchaudio

In [3]:
drums = "/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb"
with open("data/bigyuki_dataset.csv", "w", newline="") as csvfile:
    for root, dirs, files in os.walk(drums):
        for file in files:
            if file.endswith(".wav"):
                full_path = os.path.join(root, file)
                writer = csv.writer(csvfile)
                writer.writerow([full_path])
                print(full_path)

/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___english_1_r1.varispeed____1.33.wav
/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___japanese_2_r1.wav
/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___japanese_1_r1.varispeed____1.33.wav
/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___japanese_1_r1.varispeed____0.75.wav
/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___english_2_r1.wav
/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___japanese_1_snippet.varispeed____0.75.wav
/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___english_3_r1.varispeed____1.33.wav
/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102e

In [5]:
# Check the content of the CSV file, audio duration, sample rate, etc.

duration = []

with open("data/bigyuki_dataset.csv", "r") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        print(row)
        audio, sr = torchaudio.load(row[0])
        audio_duration = audio.shape[1] / sr
        duration.append(audio_duration)
        print(f"Audio duration: {audio_duration:.2f} seconds, Sample rate: {sr} Hz")

print(f"Average audio duration: {sum(duration)/len(duration):.2f} seconds")
print(f"Minimum audio duration: {min(duration):.2f} seconds")
print(f"Maximum audio duration: {max(duration):.2f} seconds")
print(f"Total duration: {sum(duration) / 3600.}")

['/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___english_1_r1.varispeed____1.33.wav']
Audio duration: 1048.19 seconds, Sample rate: 96000 Hz
['/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___japanese_2_r1.wav']
Audio duration: 2196.67 seconds, Sample rate: 96000 Hz
['/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___japanese_1_r1.varispeed____1.33.wav']
Audio duration: 854.59 seconds, Sample rate: 96000 Hz
['/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___japanese_1_r1.varispeed____0.75.wav']
Audio duration: 1522.70 seconds, Sample rate: 96000 Hz
['/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___english_2_r1.wav']
Audio duration: 848.47 seconds, Sample rate: 96000 Hz
['/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-

In [5]:
import random
train_set = []
val_set = []
test_set = []
with open("data/taiko_dataset.csv", "r") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        if random.random() < 0.8:
            train_set.append(row[0])
        elif random.random() < 0.2:
            test_set.append(row[0])
        else:
            val_set.append(row[0])
with open("data/taiko_train.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    for path in train_set:
        writer.writerow([path])
with open("data/taiko_val.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    for path in val_set:
        writer.writerow([path])
with open("data/taiko_test.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    for path in test_set:
        writer.writerow([path])

### Split data into chunks if needed

In [6]:
import torchaudio.functional as F
from pathlib import Path

# for latent diffusion bridges model

output_dir = 'data/diff_bridges/yuki'
chunk_lenght = 409600 #(17s at 24k)
sample_rate = 24000
with open("data/bigyuki_dataset.csv", "r") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        print(row)
        path = Path(str(row))
        audio, sr = torchaudio.load(row[0])
        if sr != sample_rate:
            audio = F.resample(audio, sr, sample_rate)

        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)
        total = audio.shape[1]
        n_chunks = (total - chunk_lenght) // chunk_lenght + 1
        for i in range(n_chunks):
            start = i * chunk_lenght
            chunk = audio[:, start : start + chunk_lenght]
            if chunk.shape[1] < chunk_lenght:
                break  # drop last incomplete chunk
            out = output_dir + f"/{path.stem}_chunk{i:05d}.wav"
            torchaudio.save(str(out), chunk.clamp(-1, 1), sample_rate)


['/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___english_1_r1.varispeed____1.33.wav']
['/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___japanese_2_r1.wav']
['/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___japanese_1_r1.varispeed____1.33.wav']
['/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___japanese_1_r1.varispeed____0.75.wav']
['/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___english_2_r1.wav']
['/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___japanese_1_snippet.varispeed____0.75.wav']
['/home/lois/models_benchmark/downloads/audio/22d63b9d-abb2-4c18-b3d1-9e5102edf0bb/yuki_ai_voiceover___english_3_r1.varispeed____1.33.wav']
['/home/lois/models_benchmark/downloads/audio/2